In [1]:
!pip install cplex docplex

In [2]:
import pandas as pd
from docplex.mp.model import Model

In [3]:
file_path1 = "C:/Users/kohwo/차 종류 1일 이익.xlsx"
file_path2 = "C:/Users/kohwo/방문객수.xlsx"

df_profit = pd.read_excel(file_path1, sheet_name='시트1')
df_visitor = pd.read_excel(file_path2, sheet_name='시트1')

In [4]:
df_profit.head()

,번호,차종,유종,차급,탑승인원,1일 표준대여료(원),최대보유대수(대)
0,1,모닝,휘발유,경차,5,110000,95
1,2,더 뉴 레이,휘발유,경차,5,115000,95
2,3,캐스퍼,휘발유,경차,4,120000,63
3,4,스파크,휘발유,경차,5,105000,47
4,5,모닝 어반,휘발유,경차,5,110000,47


In [5]:
df_visitor.head()

,그룹유형_k,일일_방문수,최소_필요_정원,비고
0,1인(혼행),1440,1,모든 차량 가능
1,2인(커플),1160,2,모든 차량 가능
2,3인(소가족),760,3,소형 이상 권장
3,4인(가족),400,4,중형 이상 권장
4,5인_이상(단체),240,5,승합 필수


In [6]:
I = sorted(df_profit['차급'].unique())

B = sorted(df_profit['차종'].unique())

J = sorted(df_profit['유종'].unique())

K = sorted(df_visitor['그룹유형_k'].unique())

print(I)

print(B)

print(J)

print(K)

['경차', '대형', '대형(고급)', '대형SUV', '대형SUV(고급)', '소형SUV', '승합', '전기차', '전기차(수입)', '준대형', '준중형', '준중형SUV', '중형', '중형(고급)', '중형SUV', '중형SUV(고급)']
['EV6', 'G70', 'G80', 'G90', 'GV70', 'GV80', 'K3', 'K3 GT', 'K5 (3세대)', 'K5 (LPG)', 'K5 하이브리드', 'K8', 'K8 하이브리드', 'K9', 'QM6', 'SM6', 'XM3', '그랜저 (LPG)', '그랜저 GN7', '그랜저 하이브리드', '니로 하이브리드', '더 뉴 레이', '레이 (1인승 밴)', '말리부', '모닝', '모닝 어반', '모하비', '베뉴', '셀토스', '스타리아', '스타리아 라운지', '스팅어', '스파크', '스포티지', '스포티지 하이브리드', '싼타페 MX5', '싼타페 하이브리드', '쏘나타 (LPG)', '쏘나타 디 엣지', '쏘나타 하이브리드', '쏘렌토', '쏘렌토 하이브리드', '쏠라티', '아반떼 CN7', '아반떼 하이브리드', '아이오닉5', '카니발 (4세대)', '카니발 하이리무진', '캐스퍼', '코나 (디 올 뉴)', '코나 하이브리드', '테슬라 모델3', '토레스', '투싼', '투싼 하이브리드', '트레일블레이저', '티볼리', '티볼리 에어', '팰리세이드']
['LPG', '경유', '전기', '하이브리드', '휘발유']
['1인(혼행)', '2인(커플)', '3인(소가족)', '4인(가족)', '5인_이상(단체)']


In [7]:
# 🔹 P_bij 생성 (그룹 k는 포함하지 않음)
P_bij = {}
for _, row in df_profit.iterrows():
    b = row['차종']
    i = row['차급']
    j = row['유종']
    P_bij[(b, i, j)] = row['1일 표준대여료(원)']

# -----------------------------
# c_bik 생성 (정원 충족 여부 조건 기반)
# -----------------------------
# 그룹 k를 수용하려면 차량 탑승인원 >= 그룹 최소 정원 조건
c_bik = {}

for _, car in df_profit.iterrows():
    b = car['차종']           # 브랜드
    i = car['차급']           # 차급
    capacity = car['탑승인원']  # 차량 탑승 정원

    for _, group in df_visitor.iterrows():
        k = group['그룹유형_k']           # 그룹 유형
        min_required = group['최소_필요_정원']  # 해당 그룹의 최소 필요한 정원

        # 조건 충족 시: 최소 필요 정원 값을 넣음
        if capacity >= min_required:
            c_bik[(b, i, k)] = min_required
        else:
            c_bik[(b, i, k)] = 0


# -----------------------------
# D_k 생성 (그룹별 수요: 일일 방문수)
# -----------------------------
D_k = { row['그룹유형_k']: row['일일_방문수'] for _, row in df_visitor.iterrows() }

# -----------------------------
# a_bij 생성 (차량 최대 보유대수 사용)
# -----------------------------
a_bij = {}
for _, row in df_profit.iterrows():
    b = row['차종']
    i = row['차급']
    j = row['유종']
    a_bij[(b,i,j)] = row['최대보유대수(대)']

print("P sample:", list(P_bij.items())[:5])
print("c sample:", list(c_bik.items())[:5])
print("D:", D_k)
print("a sample:", list(a_bij.items())[:5])


P sample: [(('모닝', '경차', '휘발유'), 110000), (('더 뉴 레이', '경차', '휘발유'), 115000), (('캐스퍼', '경차', '휘발유'), 120000), (('스파크', '경차', '휘발유'), 105000), (('모닝 어반', '경차', '휘발유'), 110000)]
c sample: [(('모닝', '경차', '1인(혼행)'), 1), (('모닝', '경차', '2인(커플)'), 2), (('모닝', '경차', '3인(소가족)'), 3), (('모닝', '경차', '4인(가족)'), 4), (('모닝', '경차', '5인_이상(단체)'), 5)]
D: {'1인(혼행)': 1440, '2인(커플)': 1160, '3인(소가족)': 760, '4인(가족)': 400, '5인_이상(단체)': 240}
a sample: [(('모닝', '경차', '휘발유'), 95), (('더 뉴 레이', '경차', '휘발유'), 95), (('캐스퍼', '경차', '휘발유'), 63), (('스파크', '경차', '휘발유'), 47), (('모닝 어반', '경차', '휘발유'), 47)]


In [8]:
def build_A(I, K):
    A = set()   # (i,k)만 포함

    for k in K:
        if k in ('1인(혼행)', '2인(커플)'):
            # 모든 차급 배정 가능
            for i in I:
                A.add((i, k))

        elif k == '3인(소가족)':
            # 경차 제외 (나머지는 허용)
            for i in I:
                if i != '경차':
                    A.add((i, k))

        elif k == '4인(가족)':
            # 경차, 소형SUV, 준중형, 준중형SUV 제외 → 나머지만 허용
            excluded = {'경차', '소형SUV', '준중형', '준중형SUV'}
            for i in I:
                if i not in excluded:
                    A.add((i, k))

        elif any(num in k for num in ['5','6','7','8','9']):
            for i in I:
                if i == '승합':  # 승합차만 허용
                    A.add((i,k))
    return A

# 집합 생성
A = build_A(I, K)

# 출력 확인
print("허용 (i,k) 쌍 샘플:", sorted(list(A))[:12])
print("총 허용 가능한 (i,k) 조합 수:", len(A))


허용 (i,k) 쌍 샘플: [('경차', '1인(혼행)'), ('경차', '2인(커플)'), ('대형', '1인(혼행)'), ('대형', '2인(커플)'), ('대형', '3인(소가족)'), ('대형', '4인(가족)'), ('대형(고급)', '1인(혼행)'), ('대형(고급)', '2인(커플)'), ('대형(고급)', '3인(소가족)'), ('대형(고급)', '4인(가족)'), ('대형SUV', '1인(혼행)'), ('대형SUV', '2인(커플)')]
총 허용 가능한 (i,k) 조합 수: 60


In [9]:
# -------------------------------
# 3) 모델 생성
# -------------------------------
model = Model(name="Fleet_Assignment_NoPriority")

# -------------------------------
# 4) 의사결정변수: ㅌ[b,i,j,k]
#    단, (i,k)가 A_star에 포함된 경우만 생성
# -------------------------------
x = {}
for b in B:
    for i in I:
        for j in J:
            for k in K:
                if ((b,i,j) in P_bij.keys()) and ((i,k) in A):
                    x[(b,i,j,k)] = model.integer_var(lb=0, name=f"z_{b}_{i}_{j}_{k}")

# -------------------------------
# 5) 목적 함수: 총 수익 최대화
# -------------------------------
model.maximize(model.sum(P_bij[(b,i,j)] * var for (b,i,j,k), var in x.items()))

# -------------------------------
# 6) 제약식
# -------------------------------

# (1) 공급 제약: sum(c_bik * z) ≤ D_k
for k in K:
    expr = model.sum(c_bik[(b,i,k)] * var
                 for (b,i,j,k2), var in x.items() if k2 == k)
    model.add_constraint(expr <= D_k[k], ctname=f"supply_{k}")

# (2) 보유대수 제약: sum(z[b,i,j,k]) ≤ a_bij
for b in B:
    for i in I:
        for j in J:
            expr = model.sum(var for (b2,i2,j2,k), var in x.items()
                         if (b2,i2,j2) == (b,i,j))
            if (b,i,j) in a_bij:
                model.add_constraint(expr <= a_bij[(b,i,j)], 
                                 ctname=f"cap_{b}_{i}_{j}")

# -------------------------------
# 7) 모델 풀기 및 출력
# -------------------------------
sol = model.solve(log_output=True)

if sol:
    print("Objective value:", model.objective_value)
    results=[]
    for (b,i,j,k),var in x.items():
        if var.solution_value>1e-6:
            results.append({
                '차종':b, '차급':i, '유종':j, '그룹유형':k,
                '배정대수':int(round(var.solution_value))
            })
    df_result=pd.DataFrame(results)
    display(df_result)
else:
    print("No solution found.")

Version identifier: 22.1.2.0 | 2024-12-09 | 8bd2200c8
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 0.000000 after 0.00 sec. (0.01 ticks)
Tried aggregator 2 times.
MIP Presolve eliminated 0 rows and 1 columns.
MIP Presolve modified 5 coefficients.
Aggregator did 1 substitutions.
Reduced MIP has 63 rows, 210 columns, and 420 nonzeros.
Reduced MIP has 0 binaries, 210 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (0.58 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 63 rows, 210 columns, and 420 nonzeros.
Reduced MIP has 0 binaries, 210 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (0.35 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 12 threads.
Root relaxation solution time = 0.02 sec. (0.53 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Boun

,차종,차급,유종,그룹유형,배정대수
0,EV6,전기차,전기,2인(커플),2
1,EV6,전기차,전기,3인(소가족),30
2,G70,중형(고급),휘발유,3인(소가족),24
3,G80,대형(고급),휘발유,3인(소가족),32
4,G90,대형(고급),휘발유,3인(소가족),8
5,GV70,중형SUV(고급),휘발유,3인(소가족),24
6,GV80,대형SUV(고급),휘발유,3인(소가족),16
7,K3,준중형,휘발유,1인(혼행),12
8,K3,준중형,휘발유,3인(소가족),8
9,K3 GT,준중형,휘발유,1인(혼행),16


In [10]:
df_result.to_excel(r"C:\Users\kohwo\렌터카_배정결과.xlsx", index=False)